In [1]:
import json
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

In [2]:
## creating the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en"
)

C:\Users\ansul\AppData\Local\Temp\ipykernel_10868\1333154181.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
c:\Users\ansul\OneDrive\Desktop\data science project\financial_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3740.10it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECT

In [4]:
## loading the vector store
vectorstore = FAISS.load_local(
    r"..\data\embeddings",
     embedding_model,
     allow_dangerous_deserialization=True
)

In [5]:
## loading eval dataset
with open(r"..\data\retrival_eval.json") as f:
    eval_data = json.load(f)

In [7]:
eval_data[10]

{'query': 'How do changes in currency exchange rates affect international companies?',
 'relevant_chunks': [{'company_name': 'AAPL', 'chunk_id': 19}]}

## Eval Function

In [8]:
total_queries = len(eval_data)
recall_hits = 0
mrr_total = 0

In [14]:
for item in eval_data:

    query = item["query"]
    relevant = item["relevant_chunks"]

    docs = vectorstore.similarity_search(query, k=10)

    found = False

    for rank, doc in enumerate(docs, start=1):

        meta = doc.metadata

        for rel in relevant:

            if (
                meta["company_name"] == rel["company_name"]
                and meta["chunk_id"] == rel["chunk_id"]
            ):

                if not found:
                    recall_hits += 1
                    mrr_total += 1 / rank
                    found = True

recall_at_k = recall_hits / total_queries
mrr = mrr_total / total_queries

In [16]:
print("\nEvaluation Results")
print("-------------------")
print(f"Queries: {total_queries}")
print(f"Recall@{10}: {recall_at_k:.3f}")
print(f"MRR: {mrr:.3f}")


Evaluation Results
-------------------
Queries: 20
Recall@10: 0.900
MRR: 0.477
